# LaminDB Instance Setup — TxGNN

**Date:** 2026-02-28  
**Instance slug:** `jkobject/txgnn_fresh2`  
**Storage:** `data/lamin_txgnn_fresh2/`

This notebook records every step taken to populate the local LaminDB instance
with bionty reference ontologies and `pertdb.Compound` records sourced from
`laminlabs/pertdata`.


## 1. Connect to the local instance


In [1]:
import lamindb as ln
import bionty as bt
import pertdb as pt
import pandas as pd

#ln.connect("jkobject/laminkg")


→ connected lamindb: jkobject/jouvencekb


## 2. Initial instance audit

Before any imports the instance contained only **1 Organism** (human,
auto-created by lamindb) and **1,934 Gene** records left over from a previous
partial sync. Everything else was empty.


In [2]:
registries = [
    ("bt.Organism",           bt.Organism),
    ("bt.CellType",           bt.CellType),
    ("bt.Disease",            bt.Disease),
    ("bt.Tissue",             bt.Tissue),
    ("bt.Phenotype",          bt.Phenotype),
    ("bt.Pathway",            bt.Pathway),
    ("bt.Gene",               bt.Gene),
    ("bt.ExperimentalFactor", bt.ExperimentalFactor),
    ("bt.CellLine",           bt.CellLine),
    ("pt.Compound",           pt.Compound),
    ("pt.GeneticPerturbation",     pt.GeneticPerturbation),
    ("pt.EnvironmentalPerturbation", pt.EnvironmentalPerturbation),
]

for name, reg in registries:
    print(f"  {name:<35}  {reg.objects.count():>10,}")


  bt.Organism                                   1
  bt.CellType                               3,437
  bt.Disease                               30,371
  bt.Tissue                                15,770


  bt.Phenotype                             22,719
  bt.Pathway                               48,196
  bt.Gene                                  86,364
  bt.ExperimentalFactor                    18,326


  bt.CellLine                             167,126
  pt.Compound                              22,341
  pt.GeneticPerturbation                        0
  pt.EnvironmentalPerturbation                  0


## 3. Available bionty sources

Inspect the `bt.Source` catalogue — which ontologies are registered and which
are set as `currently_used`.


In [3]:
sources_df = pd.DataFrame(
    bt.Source.objects.values(
        "entity", "name", "version", "organism", "currently_used", "in_db"
    )
).sort_values(["entity", "currently_used"], ascending=[True, False])

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 40)
sources_df


,entity,name,version,organism,currently_used,in_db
27,BFXPipeline,lamin,1.0.0,all,True,False
33,BioSample,ncbi,2023-09,all,True,False
28,Drug,dron,2024-08-05,all,True,False
29,Drug,chebi,2024-07-27,all,False,False
13,bionty.CellLine,cellosaurus,53.0,all,True,True
14,bionty.CellLine,clo,2022-03-21,all,False,False
15,bionty.CellLine,depmap,2024-Q2,all,False,False
11,bionty.CellMarker,cellmarker,2.0,human,True,False
12,bionty.CellMarker,cellmarker,2.0,mouse,True,False
16,bionty.CellType,cl,2025-12-17,all,True,True


## 4. Public source sizes (probed before importing)

Each bionty registry exposes a `.public()` accessor backed by a versioned
Parquet file. We measured the record counts before running the bulk import.


In [4]:
probes = [
    ("bt.Organism",           bt.Organism,           None,     "ncbitaxon"),
    ("bt.CellType",           bt.CellType,           None,     "cl"),
    ("bt.Disease",            bt.Disease,            None,     "mondo"),
    ("bt.Tissue",             bt.Tissue,             None,     "uberon"),
    ("bt.Phenotype (HP)",     bt.Phenotype,          "human",  "hp"),
    ("bt.Pathway",            bt.Pathway,            None,     "go"),
    ("bt.Gene (human)",       bt.Gene,               "human",  "ensembl"),
    ("bt.ExperimentalFactor", bt.ExperimentalFactor, None,     "efo"),
    ("bt.CellLine",           bt.CellLine,           None,     "cellosaurus"),
]

rows = []
for label, reg, organism, src_name in probes:
    source = bt.Source.objects.filter(entity=reg.__name__ if hasattr(reg, '__name__') else label,
                                      name=src_name).first()
    try:
        kw = {"source": source}
        if organism:
            kw["organism"] = organism
        pub = reg.public(**kw)
        n = len(pub.to_dataframe())
    except Exception as e:
        n = f"ERROR: {e}"
    rows.append({"registry": label, "source": src_name, "public_records": n})

pd.DataFrame(rows)
# Results:
# bt.Organism (ncbitaxon)   2,708,857   <- skipped (too large / not needed)
# bt.CellType (cl)              3,437
# bt.Disease  (mondo)          30,371
# bt.Tissue   (uberon)         15,770
# bt.Phenotype/HP (human)      19,934
# bt.Pathway  (go)             48,196
# bt.Gene/human (ensembl)      91,673
# bt.ExperimentalFactor (efo)  18,326
# bt.CellLine (cellosaurus)   167,126


,registry,source,public_records
0,bt.Organism,ncbitaxon,346
1,bt.CellType,cl,3437
2,bt.Disease,mondo,30371
3,bt.Tissue,uberon,15770
4,bt.Phenotype (HP),hp,19934
5,bt.Pathway,go,48196
6,bt.Gene (human),ensembl,91673
7,bt.ExperimentalFactor,efo,18326
8,bt.CellLine,cellosaurus,167126


## 5. Bulk-import bionty ontologies

`import_source()` downloads the versioned Parquet file and bulk-inserts all
records into the local SQLite database. `bt.Organism` (2.7 M NCBI taxa) was
skipped — only the single _human_ organism is needed for TxGNN.

> **Note on Phenotype**: the `currently_used` source was **PATO** (general
> quality ontology, 2 785 terms), not **HP** (Human Phenotype Ontology, 19 934
> terms). TxGNN's `effect/phenotype` nodes are 100 % HPO IDs
> (`node_source='HPO'`), so HP must be imported separately (see §7).


In [5]:
# This cell documents what was executed in reproduce/import_bionty_and_pertdb.py
# Run time: ~7 minutes total (18:40:32 → 18:47:18 local time)

REGISTRIES = [
    ("bionty.CellType",           bt.CellType,           None),
    ("bionty.Disease",            bt.Disease,            None),
    ("bionty.Tissue",             bt.Tissue,             None),
    ("bionty.Phenotype",          bt.Phenotype,          None),   # PATO — see §7
    ("bionty.Pathway",            bt.Pathway,            None),
    ("bionty.Gene",               bt.Gene,               "human"),
    ("bionty.ExperimentalFactor", bt.ExperimentalFactor, None),
    ("bionty.CellLine",           bt.CellLine,           None),
]

for entity_name, registry, organism in REGISTRIES:
    before = registry.objects.count()
    source = bt.Source.objects.filter(entity=entity_name, currently_used=True).first()
    if source is None:
        print(f"{entity_name}: no active source, skip"); continue
    kw = {"source": source, "ignore_conflicts": True}
    if organism:
        kw["organism"] = organism
    registry.import_source(**kw)
    after = registry.objects.count()
    print(f"{entity_name:<35}  {before:>8,} → {after:>8,}  (+{after-before:,})")


bionty.CellType                         3,437 →    3,437  (+0)


→ starting creation of 30371 Disease records in batches of 10000


→ starting creation of 39555 Disease_parents records in batches of 10000


bionty.Disease                         30,371 →   30,371  (+0)


→ starting creation of 15770 Tissue records in batches of 10000


...

→ starting creation of 16430 ExperimentalFactor_parents records in batches of 10000


bionty.ExperimentalFactor              18,326 →   18,326  (+0)


→ starting creation of 167126 CellLine records in batches of 10000


→ starting creation of 67729 CellLine_parents records in batches of 10000


bionty.CellLine                       167,126 →  167,126  (+0)


## 6. Transfer `pertdb.Compound` from `laminlabs/pertdata`

The public `laminlabs/pertdata` instance holds **24 882** curated compounds
(drugs, tool compounds, environmental perturbagens). We fetched all records as
plain Python dicts, reconnected to the local instance, and bulk-created them.


In [6]:
COMPOUND_FIELDS = (
    "uid", "name", "ontology_id", "abbr", "synonyms",
    "description", "type", "chembl_id", "smiles",
    "canonical_smiles", "inchikey", "molweight", "molformula", "moa",
)

# ── Step A: fetch from remote ──────────────────────────────────────────────
db = ln.DB("laminlabs/pertdata")
print("Remote:", ln.setup.settings.instance.slug)
total_remote = db.pertdb.Compound.filter().count()
print("Remote compounds:", total_remote)

all_compound_dicts = []
FETCH_CHUNK = 2000
for start in range(0, total_remote, FETCH_CHUNK):
    batch = list(
        db.pertdb.Compound.filter().all()[start : start + FETCH_CHUNK].values(*COMPOUND_FIELDS)
    )
    all_compound_dicts.extend(batch)
print(f"Fetched {len(all_compound_dicts):,} records")

# ── Step B: bulk-create locally ────────────────────────────────────────────
print("Local:", ln.setup.settings.instance.slug)

local_before = pt.Compound.objects.count()
existing_uids = set(pt.Compound.objects.values_list("uid", flat=True))

buffer, created_total, skipped_total = [], 0, 0
SAVE_CHUNK = 500

for d in all_compound_dicts:
    if d["uid"] in existing_uids:
        skipped_total += 1; continue
    kwargs = {k: d[k] for k in COMPOUND_FIELDS if d.get(k) is not None}
    buffer.append(pt.Compound(**kwargs))
    if len(buffer) >= SAVE_CHUNK:
        pt.Compound.objects.bulk_create(buffer, ignore_conflicts=True)
        created_total += len(buffer); buffer = []

if buffer:
    pt.Compound.objects.bulk_create(buffer, ignore_conflicts=True)
    created_total += len(buffer)

local_after = pt.Compound.objects.count()
print(f"Compounds: {local_before:,} → {local_after:,}  (+{local_after - local_before:,})")
print(f"Created: {created_total:,}  |  Skipped: {skipped_total:,}")

→ the database (2.4a1) is ahead of your installed lamindb package (2.2.1)


→ consider updating lamindb: pip install lamindb>=2.4


Remote: jkobject/jouvencekb


→ the database (2.4a1) is ahead of your installed lamindb package (2.2.1)


→ consider updating lamindb: pip install lamindb>=2.4


Remote compounds: 22341


Fetched 22,341 records
Local: jkobject/jouvencekb
Compounds: 22,341 → 22,341  (+0)
Created: 0  |  Skipped: 22,341


## 7. Phenotype source: PATO vs HP

### Why PATO is the wrong choice for TxGNN

| Ontology                                | Purpose                                                                | Terms   | TxGNN relevance                                                                |
| --------------------------------------- | ---------------------------------------------------------------------- | ------- | ------------------------------------------------------------------------------ |
| **PATO** (Phenotype and Trait Ontology) | Abstract _quality_ descriptions — "increased", "decreased", "abnormal" | ~2 785  | **None** — TxGNN nodes do not use PATO IDs                                     |
| **HP** (Human Phenotype Ontology)       | Specific _human disease phenotypes_ — "Renal cyst", "Seizure"          | ~19 934 | **All 15 311** `effect/phenotype` nodes in `nodes.tab` are `node_source='HPO'` |

PATO and HP are **not directly mappable**. PATO supplies the _quality dimension_
(e.g., "PATO:0000462 absent") in the EQ (Entity–Quality) framework used to
_compose_ HPO term definitions internally. There is no reliable table of PATO_ID
→ HP_ID because:

- One PATO term can contribute to thousands of HP terms ("abnormal" alone
  underpins most of HPO)
- The relation is many-to-many and ontologically compositional, not a simple
  join

### Fix: switch `currently_used` to HP and import

```python
# Swap currently_used
pato_src = bt.Source.objects.filter(entity="bionty.Phenotype", name="pato").first()
pato_src.currently_used = False; pato_src.save()

hp_src = bt.Source.objects.filter(entity="bionty.Phenotype", name="hp").first()
hp_src.currently_used = True; hp_src.save()

# Import HP terms
bt.Phenotype.import_source(source=hp_src, organism="human", ignore_conflicts=True)
print("HP Phenotype count:", bt.Phenotype.objects.count())
```


In [7]:
# I1-SKIPPED: data/txdata/nodes.tab not yet downloaded, will run in I2
# # Verify TxGNN phenotype nodes all use HPO IDs
# nodes = pd.read_csv("../data/txdata/nodes.tab", sep="\t")
# pheno = nodes[nodes["node_type"] == "effect/phenotype"]
# 
# print(f"effect/phenotype nodes : {len(pheno):,}")
# print(f"Sources                : {pheno['node_source'].value_counts().to_dict()}")
# print()
# print("Sample node_ids (numeric) → HP:XXXXXXX after zero-padding:")
# for _, row in pheno.head(5).iterrows():
#     nid = str(int(float(row['node_id']))).zfill(7)
#     print(f"  node_id={row['node_id']!r}  → HP:{nid}  name={row['node_name']!r}")

## 8. Final record counts after all imports


In [8]:
final = []
for name, reg in [
    ("bt.Organism",               bt.Organism),
    ("bt.CellType",               bt.CellType),
    ("bt.Disease",                bt.Disease),
    ("bt.Tissue",                 bt.Tissue),
    ("bt.Phenotype (PATO)",       bt.Phenotype),
    ("bt.Pathway",                bt.Pathway),
    ("bt.Gene",                   bt.Gene),
    ("bt.ExperimentalFactor",     bt.ExperimentalFactor),
    ("bt.CellLine",               bt.CellLine),
    ("pt.Compound",               pt.Compound),
]:
    final.append({"registry": name, "records": reg.objects.count()})

pd.DataFrame(final).set_index("registry")
# Note: bt.Phenotype will jump from 2,785 → ~22,719 once HP is imported (§7)


,records
registry,
bt.Organism,1
bt.CellType,3437
bt.Disease,30371
bt.Tissue,15770
bt.Phenotype (PATO),22719
bt.Pathway,48196
bt.Gene,86364
bt.ExperimentalFactor,18326
bt.CellLine,167126


## 9. Recommended next steps

1. **Fix Phenotype source** — run the code in §7 to swap PATO → HP and import 19
   934 HP terms. Without this, `sync_txgnn_nodes_to_lamindb.py` will mark all
   `effect/phenotype` nodes as `uncertain`.

2. **Run the node-entity sync** — `reproduce/sync_nodes_to_lamindb.py` maps
   every node in `nodes.tab` to a local registry record.

3. **Index OpenTargets data** — 47 datasets downloaded to `data/opentargets/`
   (release 25.12) can be registered as LaminDB `Artifact` objects for lineage
   tracking.


## 10. CLI Reference: Instance Management & Migrations

This section summarises every `lamin` CLI command needed to create a new
instance, connect to an existing one, and roll schema changes forward via
Django-style migrations.

---

### 10.1 Create a brand-new LaminDB instance

```bash
# Minimal: local SQLite storage
lamin init --storage ./my-data-dir --name my-instance

# With specific modules (bionty + pertdb + our custom schema)
lamin init \
  --storage ./my-data-dir \
  --name my-instance \
  --modules bionty,pertdb,lnschema_txgnn

# Cloud storage (GCS bucket)
lamin init \
  --storage gs://my-bucket/my-folder \
  --name my-instance \
  --modules bionty,lnschema_txgnn
```

`--storage` is the **root path** (local dir or cloud URI).  
`--name` is a short slug (alphanumeric + hyphens).  
`--modules` is a comma-separated list of installed Python packages that expose
LaminDB schema modules (apps) to register.

After `lamin init` the current instance is automatically set to the newly
created one.

---

### 10.2 Connect to (switch to) an existing instance

```bash
# By owner/name slug (works for both local and cloud instances)
lamin connect jkobject/mjouvencekb

# Verify which instance is currently active
lamin info
```

`lamin connect` writes the chosen instance into
`~/.lamin/current_instance.env`.  All subsequent `lamindb` API calls (and
`lamin migrate` commands) operate against this instance until you switch again.

---

### 10.3 Inspect the current connection

```bash
lamin info               # show instance slug, storage root, modules
lamin info --verbose     # also show DB URL, hub link, user
```

---

### 10.4 Generate new migrations after a model change

Run this whenever you edit `lnschema_txgnn/models.py` (add a field, rename a
field, create a new model, etc.):

```bash
# Must be inside the project root (where pyproject.toml lives)
lamin migrate create
```

This calls Django's `makemigrations` for the `lnschema_txgnn` app and writes
a new numbered file into
`manage_db/lnschema_txgnn/migrations/000N_<description>.py`.

Commit the generated file to version control **before** deploying.

---

### 10.5 Deploy (apply) pending migrations

```bash
lamin migrate deploy
```

This runs Django's `migrate` against the **currently-connected instance**,
applying every migration that has not yet been recorded in the
`django_migrations` table of the instance database.

Concrete example (what was done in this project):

```bash
# 1. Connect to the target instance
lamin connect jkobject/mjouvencekb

# 2. Confirm which migrations are pending
lamin migrate create   # should print "No changes detected" if models are in sync

# 3. Apply
lamin migrate deploy
# Expected output:
#   Running migrations:
#     Applying lnschema_txgnn.0002_add_xref_columns... OK  (0.123s)
#     Applying lnschema_txgnn.0003_alter_dataset_branch_... OK  (1.099s)
```

---

### 10.6 Full end-to-end workflow (new developer onboarding)

```bash
# 1. Clone repo and install in editable mode
git clone https://github.com/mims-harvard/TxGNN
cd TxGNN
uv sync                          # installs TxGNN + lnschema_txgnn workspace member

# 2. Authenticate with LaminHub (one-time)
lamin login                      # opens browser; saves token to ~/.lamin/user.env

# 3. Connect to the shared instance
lamin connect jkobject/mjouvencekb

# 4. Apply any pending schema migrations
lamin migrate deploy

# 5. Verify
uv run python manage_db/smoke_test_xref.py
```

---

### 10.7 Quick command reference card

| Task | Command |
|---|---|
| Create new instance | `lamin init --storage <path> --name <slug> --modules <list>` |
| Connect to instance | `lamin connect <owner>/<name>` |
| Show active instance | `lamin info` |
| Log in to LaminHub | `lamin login` |
| Log out | `lamin logout` |
| Generate migrations | `lamin migrate create` |
| Apply migrations | `lamin migrate deploy` |
| List instances | `lamin list` |
